# Controller 后训练 Runbook

这个 notebook 用来在 Jupyter 里跑当前主线：`prepare_round -> SFT -> GRPO -> evaluate -> annotate_queue -> next_round`。

先改下面配置单元里的 `BASE_MODEL`。SFT/GRPO 是长任务，默认用开关控制。

In [ ]:
from __future__ import annotations

import json
import os
import sys
from pathlib import Path

def find_repo_root(start: Path) -> Path:
    for candidate in [start.resolve(), *start.resolve().parents]:
        if (candidate / "src" / "task_router_graph_train").exists():
            return candidate
    fallback = Path("/root/WORK/task-rounting").resolve()
    if (fallback / "src" / "task_router_graph_train").exists():
        return fallback
    raise FileNotFoundError("找不到仓库根目录；请从 repo 内启动 Jupyter")


REPO_ROOT = find_repo_root(Path.cwd())

SRC_ROOT = REPO_ROOT / "src"
if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))

print("REPO_ROOT =", REPO_ROOT)
print("PYTHONPATH has src =", str(SRC_ROOT) in sys.path)

In [ ]:
# ===== 基础配置 =====
ROUND_ID = "round_0001"
PREVIOUS_ROUND_ID = ""  # 例如下一轮填 "round_0001"

# 可以填本地模型目录，或 HuggingFace model id。
BASE_MODEL = os.environ.get("BASE_MODEL", "")

# 常见 Qwen/Llama 结构先用 q_proj/v_proj；如果你的模型模块名不同，在这里改。
LORA_TARGET_MODULES = ["q_proj", "v_proj"]

SFT_OUTPUT_DIR = REPO_ROOT / "var" / "runs" / "task_router_graph_train" / "sft" / ROUND_ID
GRPO_OUTPUT_DIR = REPO_ROOT / "var" / "runs" / "task_router_graph_train" / "grpo" / ROUND_ID
EVAL_OUTPUT_DIR = REPO_ROOT / "var" / "runs" / "task_router_graph_train" / "eval" / ROUND_ID

# 长任务开关
RUN_SFT = False
RUN_GRPO_EXPORT_ONLY = True
RUN_GRPO_FULL_UPDATE = False
RUN_EVALUATE = False
RUN_ANNOTATE_QUEUE = False

# evaluate 需要你先用模型对 holdout_records.jsonl 生成 predictions.jsonl。
HOLDOUT_PREDICTIONS = REPO_ROOT / "var" / "runs" / "task_router_graph_train" / "predictions" / f"{ROUND_ID}_holdout_predictions.jsonl"

print(json.dumps({
    "ROUND_ID": ROUND_ID,
    "PREVIOUS_ROUND_ID": PREVIOUS_ROUND_ID,
    "BASE_MODEL": BASE_MODEL,
    "SFT_OUTPUT_DIR": str(SFT_OUTPUT_DIR),
    "GRPO_OUTPUT_DIR": str(GRPO_OUTPUT_DIR),
    "EVAL_OUTPUT_DIR": str(EVAL_OUTPUT_DIR),
    "HOLDOUT_PREDICTIONS": str(HOLDOUT_PREDICTIONS),
}, ensure_ascii=False, indent=2))

## 1. Prepare Round

生成当前 round 的 SFT、GRPO、holdout、teacher queue 等资产。

In [ ]:
from task_router_graph_train.dataset import prepare_round_assets

prepare_report = prepare_round_assets(
    round_id=ROUND_ID,
    previous_round_id=PREVIOUS_ROUND_ID.strip() or None,
    workspace_root=REPO_ROOT,
)

print(json.dumps(prepare_report, ensure_ascii=False, indent=2))

## 2. SFT

把 `RUN_SFT` 改成 `True` 后执行。第一次建议小 epoch 跑通链路，再加训练量。

In [ ]:
from task_router_graph_train.train import train_controller_sft

if RUN_SFT:
    if not BASE_MODEL:
        raise ValueError("请先在配置单元设置 BASE_MODEL，或设置环境变量 BASE_MODEL")

    sft_report = train_controller_sft(
        round_id=ROUND_ID,
        model_name_or_path=BASE_MODEL,
        lora_target_modules=LORA_TARGET_MODULES,
        output_dir=SFT_OUTPUT_DIR,
        num_train_epochs=5,
        per_device_train_batch_size=1,
        gradient_accumulation_steps=4,
        learning_rate=2e-4,
        max_seq_length=2048,
        lora_r=8,
        lora_alpha=16,
        lora_dropout=0.05,
        seed=42,
    )
    print(json.dumps(sft_report, ensure_ascii=False, indent=2))
else:
    print("跳过 SFT。需要训练时把 RUN_SFT 改成 True。")

## 3. GRPO

`RUN_GRPO_EXPORT_ONLY=True` 只导出 verl 数据和请求文件，不启动训练。真正训练需要安装 `verl`，并设置 teacher API key，例如 `os.environ["API_KEY_Qwen"] = "..."`。

In [ ]:
from task_router_graph_train.cli.train_grpo import DEFAULT_GRPO_CONFIG_PATH
from task_router_graph_train.train import train_controller_grpo

if RUN_GRPO_EXPORT_ONLY or RUN_GRPO_FULL_UPDATE:
    if not BASE_MODEL:
        raise ValueError("请先在配置单元设置 BASE_MODEL，或设置环境变量 BASE_MODEL")

    grpo_report = train_controller_grpo(
        round_id=ROUND_ID,
        config_path=DEFAULT_GRPO_CONFIG_PATH,
        output_dir=GRPO_OUTPUT_DIR,
        runtime_root=REPO_ROOT,
        model_name_or_path=BASE_MODEL,
        lora_target_modules=LORA_TARGET_MODULES,
        num_candidates=4,
        seed=42,
        num_train_epochs=1,
        per_device_train_batch_size=1,
        gradient_accumulation_steps=4,
        learning_rate=2e-4,
        max_seq_length=2048,
        lora_r=8,
        lora_alpha=16,
        lora_dropout=0.05,
        export_only=not RUN_GRPO_FULL_UPDATE,
    )
    print(json.dumps(grpo_report, ensure_ascii=False, indent=2))
else:
    print("跳过 GRPO。需要导出数据时把 RUN_GRPO_EXPORT_ONLY 改成 True；需要训练时把 RUN_GRPO_FULL_UPDATE 改成 True。")

## 4. Holdout Evaluate

先用当前模型对 round 的 `holdout_records.jsonl` 生成 `predictions.jsonl`，再执行这个单元。失败样本可以自动进入当前 round 的 `teacher_queue.jsonl`。

In [ ]:
from task_router_graph_train.eval import evaluate_holdout_predictions
from task_router_graph_train.rounds import load_round_manifest, resolve_round_asset_path

manifest = load_round_manifest(round_id=ROUND_ID)
holdout_records = resolve_round_asset_path(manifest, "holdout_records")
print("holdout_records =", holdout_records)
print("predictions =", HOLDOUT_PREDICTIONS)

if RUN_EVALUATE:
    if not HOLDOUT_PREDICTIONS.exists():
        raise FileNotFoundError(f"缺少 predictions.jsonl: {HOLDOUT_PREDICTIONS}")

    eval_report = evaluate_holdout_predictions(
        record_path=holdout_records,
        prediction_path=HOLDOUT_PREDICTIONS,
        enqueue_failed_badcases=True,
        badcase_round_manifest=Path(manifest["_manifest_path"]),
    )
    EVAL_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    (EVAL_OUTPUT_DIR / "metrics_summary.json").write_text(
        json.dumps(eval_report["metrics_summary"], ensure_ascii=False, indent=2) + "\n",
        encoding="utf-8",
    )
    (EVAL_OUTPUT_DIR / "evidence_rows.jsonl").write_text(
        "\n".join(json.dumps(row, ensure_ascii=False) for row in eval_report["evidence_rows"]) + "\n",
        encoding="utf-8",
    )
    print(json.dumps(eval_report["metrics_summary"], ensure_ascii=False, indent=2))
else:
    print("跳过 evaluate。准备好 predictions 后把 RUN_EVALUATE 改成 True。")

## 5. Annotate Queue

把 `teacher_queue.jsonl` 里的 badcase 交给 teacher，接纳的样本写入 `sft_admissions.jsonl`。下一轮 `prepare_round` 用 `PREVIOUS_ROUND_ID` 合入。

In [ ]:
from task_router_graph_train.feedback import annotate_teacher_queue

if RUN_ANNOTATE_QUEUE:
    annotation_report = annotate_teacher_queue(round_id=ROUND_ID)
    print(json.dumps(annotation_report, ensure_ascii=False, indent=2))
else:
    print("跳过 annotate_queue。需要回流 SFT 样本时把 RUN_ANNOTATE_QUEUE 改成 True。")

## 6. 下一轮

下一轮把配置改成：

```python
ROUND_ID = "round_0002"
PREVIOUS_ROUND_ID = "round_0001"
```

然后从 `Prepare Round` 重新执行。